<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Finance (2nd ed.)

**Mastering Data-Driven Finance**

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH

<img src="http://hilpisch.com/images/py4fi_2nd_shadow.png" width="300px" align="left">

# Model Calibration

## The Data

In [ ]:
import numpy as np
import pandas as pd
import datetime as dt

In [ ]:
from pylab import mpl, plt
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'
%config InlineBackend.figure_format = 'svg'

In [ ]:
import sys
sys.path.append('../')
sys.path.append('../dx')

In [ ]:
dax = pd.read_csv('../../source/tr_eikon_option_data.csv',
                 index_col=0)  

In [ ]:
for col in ['CF_DATE', 'EXPIR_DATE']:
    dax[col] = dax[col].apply(lambda date: pd.Timestamp(date))  

In [ ]:
dax.info()  

In [ ]:
dax.set_index('Instrument').head(7)  

In [ ]:
initial_value = dax.iloc[0]['CF_CLOSE']  

In [ ]:
calls = dax[dax['PUTCALLIND'] == 'CALL'].copy()  
puts = dax[dax['PUTCALLIND'] == 'PUT '].copy()  

In [ ]:
calls.set_index('STRIKE_PRC')[['CF_CLOSE', 'IMP_VOLT']].plot(
    secondary_y='IMP_VOLT', style=['bo', 'rv'], figsize=(10, 6));
# plt.savefig('../../images/ch21/dx_cal_01.png');

In [ ]:
ax = puts.set_index('STRIKE_PRC')[['CF_CLOSE', 'IMP_VOLT']].plot(
    secondary_y='IMP_VOLT', style=['bo', 'rv'], figsize=(10, 6))
ax.get_legend().set_bbox_to_anchor((0.25, 0.5));
# plt.savefig('../../images/ch21/dx_cal_02.png');

## Model Calibration

### Relevant Market Data

In [ ]:
limit = 500  

In [ ]:
option_selection = calls[abs(calls['STRIKE_PRC'] - initial_value) < limit].copy()  

In [ ]:
option_selection.info()  

In [ ]:
option_selection.set_index('Instrument').tail()  

In [ ]:
option_selection.set_index('STRIKE_PRC')[['CF_CLOSE', 'IMP_VOLT']].plot(
    secondary_y='IMP_VOLT', style=['bo', 'rv'], figsize=(10, 6));
# plt.savefig('../../images/ch21/dx_cal_03.png');

### Option Modeling

In [ ]:
from valuation_mcs_european import valuation_mcs_european
from jump_diffusion import jump_diffusion
from market_environment import market_environment
from constant_short_rate import constant_short_rate
from derivatives_position import derivatives_position
from derivatives_portfolio import derivatives_portfolio

In [ ]:
pricing_date = option_selection['CF_DATE'].max()  

In [ ]:
me_dax = market_environment('DAX30', pricing_date)  

In [ ]:
maturity = pd.Timestamp(calls.iloc[0]['EXPIR_DATE'])  

In [ ]:
me_dax.add_constant('initial_value', initial_value)  
me_dax.add_constant('final_date', maturity)  
me_dax.add_constant('currency', 'EUR')  

In [ ]:
me_dax.add_constant('frequency', 'B')  
me_dax.add_constant('paths', 10000)  

In [ ]:
csr = constant_short_rate('csr', 0.01)  
me_dax.add_curve('discount_curve', csr)  

In [ ]:
me_dax.add_constant('volatility', 0.2)
me_dax.add_constant('lambda', 0.8)
me_dax.add_constant('mu', -0.2)
me_dax.add_constant('delta', 0.1)

In [ ]:
dax_model = jump_diffusion('dax_model', me_dax)

In [ ]:
me_dax.add_constant('strike', initial_value)  
me_dax.add_constant('maturity', maturity)

In [ ]:
payoff_func = 'np.maximum(maturity_value - strike, 0)'  

In [ ]:
dax_eur_call = valuation_mcs_european('dax_eur_call',
                        dax_model, me_dax, payoff_func)  

In [ ]:
dax_eur_call.present_value()  

In [ ]:
option_models = {}  
for option in option_selection.index:
    strike = option_selection['STRIKE_PRC'].loc[option]  
    me_dax.add_constant('strike', strike)  
    option_models[strike] = valuation_mcs_european(
                                'eur_call_%d' % strike,
                                dax_model,
                                me_dax,
                                payoff_func)

In [ ]:
def calculate_model_values_old(p0):
    ''' Returns all relevant option values.
    
    Parameters
    ===========
    p0: tuple/list
        tuple of kappa, theta, volatility
    
    Returns
    =======
    model_values: dict
        dictionary with model values
    '''
    volatility, lamb, mu, delta = p0
    dax_model.update(volatility=volatility, lamb=lamb, mu=mu, delta=delta)
    model_values = {}
    for strike in option_models:
        model_values[strike] = option_models[strike].present_value(fixed_seed=True)
    return model_values

In [ ]:
def calculate_model_values(p0):
    ''' Returns all relevant option values.
    
    Parameters
    ===========
    p0: tuple/list
        tuple of kappa, theta, volatility
    
    Returns
    =======
    model_values: dict
        dictionary with model values
    '''
    volatility, lamb, mu, delta = p0
    dax_model.update(volatility=volatility, lamb=lamb,
                     mu=mu, delta=delta)
    return {
            strike: model.present_value(fixed_seed=True)
            for strike, model in option_models.items()
        }

In [ ]:
calculate_model_values((0.1, 0.1, -0.4, 0.0))

### Calibration Procedure

In [ ]:
i = 0
def mean_squared_error(p0):
    ''' Returns the mean-squared error given
    the model and market values.
    
    Parameters
    ===========
    p0: tuple/list
        tuple of kappa, theta, volatility
    
    Returns
    =======
    MSE: float
        mean-squared error
    '''
    global i
    model_values = np.array(list(calculate_model_values(p0).values()))  
    market_values = option_selection['CF_CLOSE'].values  
    option_diffs = model_values - market_values  
    MSE = np.sum(option_diffs ** 2) / len(option_diffs)  
    if i % 75 == 0:
        if i == 0:
            print('%4s  %6s  %6s  %6s  %6s --> %6s' % 
                 ('i', 'vola', 'lambda', 'mu', 'delta', 'MSE'))
        print('%4d  %6.3f  %6.3f  %6.3f  %6.3f --> %6.3f' % 
                (i, p0[0], p0[1], p0[2], p0[3], MSE))
    i += 1
    return MSE        

In [ ]:
mean_squared_error((0.1, 0.1, -0.4, 0.0))  

In [ ]:
import scipy.optimize as spo

In [ ]:
%%time
i = 0
opt_global = spo.brute(mean_squared_error,
                  ((0.10, 0.201, 0.025),  # range for volatility
                   (0.10, 0.80, 0.10),  # range for jump intensity
                   (-0.40, 0.01, 0.10),  # range for average jump size
                   (0.00, 0.121, 0.02)),  # range for jump variability
                 finish=None)

In [ ]:
mean_squared_error(opt_global)

In [ ]:
%%time
i = 0
opt_local = spo.fmin(mean_squared_error, opt_global,
                     xtol=0.00001, ftol=0.00001,
                     maxiter=200, maxfun=550)

In [ ]:
i = 0
mean_squared_error(opt_local)  

In [ ]:
calculate_model_values(opt_local)  

In [ ]:
option_selection['MODEL'] = np.array(list(calculate_model_values(opt_local).values()))
option_selection['ERRORS_EUR'] = (option_selection['MODEL'] -
                                  option_selection['CF_CLOSE'])
option_selection['ERRORS_%'] = (option_selection['ERRORS_EUR'] /
                                option_selection['CF_CLOSE']) * 100

In [ ]:
option_selection[['MODEL', 'CF_CLOSE', 'ERRORS_EUR', 'ERRORS_%']]

In [ ]:
round(option_selection['ERRORS_EUR'].mean(), 3)  

In [ ]:
round(option_selection['ERRORS_%'].mean(), 3)  

In [ ]:
fix, (ax1, ax2, ax3) = plt.subplots(3, sharex=True, figsize=(10, 10))
strikes = option_selection['STRIKE_PRC'].values
ax1.plot(strikes, option_selection['CF_CLOSE'], label='market quotes')
ax1.plot(strikes, option_selection['MODEL'], 'ro', label='model values')
ax1.set_ylabel('option values')
ax1.legend(loc=0)
wi = 15
ax2.bar(strikes - wi / 2., option_selection['ERRORS_EUR'], width=wi)
ax2.set_ylabel('errors [EUR]')
ax3.bar(strikes - wi / 2., option_selection['ERRORS_%'], width=wi)
ax3.set_ylabel('errors [%]')
ax3.set_xlabel('strikes');

## Market-Based Valuation

### Modeling Option Positions

In [ ]:
me_dax = market_environment('me_dax', pricing_date)
me_dax.add_constant('initial_value', initial_value)
me_dax.add_constant('final_date', pricing_date)
me_dax.add_constant('currency', 'EUR')

In [ ]:
me_dax.add_constant('volatility', opt_local[0])  
me_dax.add_constant('lambda', opt_local[1])  
me_dax.add_constant('mu', opt_local[2])  
me_dax.add_constant('delta', opt_local[3])  

In [ ]:
me_dax.add_constant('model', 'jd')

In [ ]:
payoff_func = 'np.maximum(strike - instrument_values, 0)'

In [ ]:
shared = market_environment('share', pricing_date)  
shared.add_constant('maturity', maturity)  
shared.add_constant('currency', 'EUR')  

In [ ]:
option_positions = {}
option_environments = {}
for option in option_selection.index:
    option_environments[option] = market_environment(
        'am_put_%d' % option, pricing_date)  
    strike = option_selection['STRIKE_PRC'].loc[option]  
    option_environments[option].add_constant('strike', strike)  
    option_environments[option].add_environment(shared)  
    option_positions['am_put_%d' % strike] = \
                    derivatives_position(
                        'am_put_%d' % strike,
                        quantity=np.random.randint(10, 50),
                        underlying='dax_model',
                        mar_env=option_environments[option],
                        otype='American',
                        payoff_func=payoff_func)  

### The Options Portfolio

In [ ]:
val_env = market_environment('val_env', pricing_date)
val_env.add_constant('starting_date', pricing_date)
val_env.add_constant('final_date', pricing_date)  
val_env.add_curve('discount_curve', csr)
val_env.add_constant('frequency', 'B')
val_env.add_constant('paths', 25000)

In [ ]:
underlyings = {'dax_model' : me_dax}  

In [ ]:
portfolio = derivatives_portfolio('portfolio', option_positions,
                                  val_env, underlyings)  

In [ ]:
%time results = portfolio.get_statistics(fixed_seed=True)

In [ ]:
results.round(1)

In [ ]:
results[['pos_value','pos_delta','pos_vega']].sum().round(1)

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>